In [1]:
# re = módulo de regex nativo do Python.
import re

In [2]:
# Texto com dois 'Paul' de pessoas diferentes e um 'Paul' sozinho.
text = "Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common."

In [3]:
# Regex: 'Paul' + espaço + letra maiúscula + resto da palavra. Pega 'Paul Newman' e 'Paul
# Hollywood', mas ignora 'Paul is'.
pattern = r"Paul [A-Z]\w+"

In [4]:
# finditer devolve cada ocorrência como objeto Match (com posição e texto).
matches = re.finditer(pattern, text)
for match in matches:
    print (match)

<re.Match object; span=(0, 11), match='Paul Newman'>
<re.Match object; span=(39, 53), match='Paul Hollywood'>


In [5]:
# Span = trecho de um doc, que pode virar uma entidade.
import spacy
from spacy.tokens import Span

INFO:tensorflow:Enabling eager execution
INFO:tensorflow:Enabling v2 tensorshape
INFO:tensorflow:Enabling resource variables
INFO:tensorflow:Enabling tensor equality
INFO:tensorflow:Enabling control flow v2


In [11]:
# Transforma os matches do regex em entidades PERSON, na mão, sem nenhum modelo.
# Pipeline vazio: sem NER, então doc.ents começa vazio.
nlp = spacy.blank("en")
doc = nlp(text)
print (doc.ents)
# Copia as entidades que já existem (aqui nenhuma).
original_ents = list(doc.ents)
mwt_ents = []
# O regex trabalha em posições de CARACTERES, mas doc.ents trabalha com índices de TOKENS.
for match in re.finditer(pattern, doc.text):
    start, end = match.span()
    # char_span converte posição de caractere em Span de tokens (devolve None se não bater com
    # limites de token).
    span = doc.char_span(start, end)
    if span is not None:
        mwt_ents.append((span.start, span.end, span.text))
for ent in mwt_ents:
    start, end, name = ent
    # Cria a entidade com o rótulo PERSON.
    per_ent = Span(doc, start, end, label="PERSON")
    original_ents.append(per_ent)
# Atribui a lista final (os spans não podem se sobrepor).
doc.ents = original_ents
for ent in doc.ents:
    print (ent.text, ent.label_)

()
Paul Newman PERSON
Paul Hollywood PERSON


In [8]:
# mwt_ents guarda (token inicial, token final, texto) de cada match.
print (mwt_ents)

[(0, 2, 'Paul Newman'), (8, 10, 'Paul Hollywood')]


In [14]:
# A mesma lógica da célula 5, agora empacotada como componente de pipeline ('paul_ner').
from spacy.language import Language

@Language.component("paul_ner")
def paul_ner(doc):
    # Regex dentro do componente.
    pattern = r"Paul [A-Z]\w+"
    original_ents = list(doc.ents)
    mwt_ents = []
    # Acha os matches no texto cru (caracteres).
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        # Converte pra Span de tokens.
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))
    for ent in mwt_ents:
        start, end, name = ent
        # Cria a entidade PERSON.
        per_ent = Span(doc, start, end, label="PERSON")
        original_ents.append(per_ent)
    doc.ents = original_ents
    # Devolve o doc com as entidades atualizadas.
    return (doc)

In [15]:
# Novo pipeline em branco com o componente paul_ner (adicionado pelo nome registrado).
nlp2 = spacy.blank("en")
nlp2.add_pipe("paul_ner")

<function __main__.paul_ner(doc)>

In [16]:
# Deve mostrar as duas entidades 'Paul ...'.
doc2 = nlp2(text)
print (doc2.ents)

(Paul Newman, Paul Hollywood)


In [20]:
# Outro componente: marca 'Hollywood' como CINEMA. Usa filter_spans pra resolver sobreposição.
from spacy.language import Language
# filter_spans fica com o span mais longo quando há sobreposição (sem isso, atribuir doc.ents com
# spans sobrepostos dá erro).
from spacy.util import filter_spans
@Language.component("cinema_ner")
def cinema_ner(doc):
    pattern = r"Hollywood"
    original_ents = list(doc.ents)
    mwt_ents = []
    for match in re.finditer(pattern, doc.text):
        start, end = match.span()
        span = doc.char_span(start, end)
        if span is not None:
            mwt_ents.append((span.start, span.end, span.text))
    for ent in mwt_ents:
        start, end, name = ent
        per_ent = Span(doc, start, end, label="CINEMA")
        original_ents.append(per_ent)
    # Descarta os spans que se sobrepõem.
    filtered = filter_spans(original_ents)
    doc.ents = filtered
    return (doc)

In [21]:
# Aqui usa o modelo sm (com NER treinado) e adiciona o cinema_ner depois dele.
nlp3 = spacy.load("en_core_web_sm")
nlp3.add_pipe("cinema_ner")

<function __main__.cinema_ner(doc)>

In [22]:
# Se o modelo marcou 'Paul Hollywood' como PERSON, o 'Hollywood' sozinho é descartado pelo
# filter_spans (fica o span mais longo).
doc3 = nlp3(text)
for ent in doc3.ents:
    print (ent.text, ent.label_)

Paul Newman PERSON
American NORP
Paul Hollywood PERSON
British NORP
Paul PERSON
